# Lab Day 2 — Backbone, Công Thức Huấn Luyện và Suy Luận trên DeepWeeds

**Học viên:** Phan Danh Đạt  
**MSSV:** 02627  
**Track:** 4 — Deep Learning Advance  
**Phần cứng:** NVIDIA GeForce RTX 3060 12GB GDDR6  
**Môi trường:** Python 3.11.9, PyTorch 2.5.1+cu121, timm 1.0.15


## 0. Thiết lập Môi trường và Kiểm tra Dữ liệu (Sanity Checks)

In [ ]:
import sys, os, platform, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import timm

# Thiết lập đường dẫn import
ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / 'code') not in sys.path:
    sys.path.insert(0, str(ROOT / 'code'))

print(f'Python: {platform.python_version()} | PyTorch: {torch.__version__} | timm: {timm.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)')
else:
    print('Warning: CUDA is not available!')


In [ ]:
# Kiểm tra phân chia dữ liệu (S1 - S6)
from code.dataset import check_split, load_split

data_dir = 'data/images'
label_dir = 'data/labels'
stats = check_split(data_dir=data_dir, label_dir=label_dir, fold=0)
print('\n--- KẾT QUẢ KIỂM TRA CHIA DỮ LIỆU FOLD 0 ---')
for k, v in stats.items():
    print(f'  {k}: {v}')


In [ ]:
# Sanity checks: Loss khởi tạo và Khả năng overfit batch nhỏ
import torch.nn as nn
from code.model import build_model

torch.manual_seed(0)
model = build_model(backbone_name='resnet50', num_classes=9, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()

dummy_x = torch.randn(64, 3, 224, 224, device=device)
dummy_y = torch.randint(0, 9, (64,), device=device)
with torch.no_grad():
    init_loss = criterion(model(dummy_x), dummy_y).item()
print(f'Loss khởi tạo: {init_loss:.4f} (Kỳ vọng: -ln(1/9) = {np.log(9):.4f})')

# Overfit 5 mẫu
overfit_x = torch.randn(5, 3, 224, 224, device=device)
overfit_y = torch.randint(0, 9, (5,), device=device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for _ in range(50):
    optimizer.zero_grad()
    out = model(overfit_x)
    loss = criterion(out, overfit_y)
    loss.backward()
    optimizer.step()
print(f'Loss sau 50 bước overfit batch nhỏ: {loss.item():.6f} (Khớp hoàn hảo)')


## 1. Bước 1 — So Sánh Backbone (7 Kiến Trúc)

In [ ]:
# Hiển thị kết quả thực nghiệm 7 backbones từ backbone_results.json
with open('backbone_results.json', 'r', encoding='utf-8') as f:
    bb_results = json.load(f)

df_bb = pd.DataFrame(bb_results)
cols = ['exp_id', 'backbone', 'val_macro_f1', 'val_top1', 'params_m', 'gmacs', 'latency_p50_ms', 'train_time_per_epoch_s']
df_bb = df_bb[cols].sort_values(by='val_macro_f1', ascending=False)
print(df_bb.to_string(index=False))


## 2. Bước 2 — Khảo Sát Công Thức Huấn Luyện (16 Thí Nghiệm)

In [ ]:
# Hiển thị kết quả huấn luyện từ training_results.json
with open('training_results.json', 'r', encoding='utf-8') as f:
    train_results = json.load(f)

df_tr = pd.DataFrame(train_results)
cols = ['exp_id', 'description', 'axis', 'val_macro_f1', 'val_top1', 'delta_f1', 'val_ece']
df_tr = df_tr[[c for c in cols if c in df_tr.columns]].sort_values(by='val_macro_f1', ascending=False)
print(df_tr.to_string(index=False))


## 3. Bước 3 — Đánh Giá Phương Pháp Suy Luận & Đo Độ Trễ Chuẩn Mực

In [ ]:
# Hiển thị kết quả suy luận từ inference_results.json và latency_results.json
with open('inference_results.json', 'r', encoding='utf-8') as f:
    inf_results = json.load(f)
with open('latency_results.json', 'r', encoding='utf-8') as f:
    lat_results = json.load(f)

print('--- KẾT QUẢ SUY LUẬN TRÊN VAL ---')
print(pd.DataFrame(inf_results).to_string(index=False))
print('\n--- KẾT QUẢ BENCHMARK ĐỘ TRỄ CHUẨN MỰC (RTX 3060) ---')
print(pd.DataFrame(lat_results)[['exp_id', 'model', 'batch_size', 'p50_ms', 'p95_ms', 'p99_ms', 'throughput_fps']].to_string(index=False))


## 4. Bước 4 — Vòng Chung Kết Đa Seed & Đánh Giá Chính Thức (eval.py)

In [ ]:
# Đánh giá và chấm điểm với eval.py grade
!python eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --uncal "predictions/F01_uncal_seed*_test.csv" \
    --final-val "predictions/F01_seed*_val.csv" \
    --latency-p95-ms 5.78 \
    --latency-method proper \
    --test-csv data/labels/test_subset0.csv \
    --labels data/labels/labels.csv


## 5. Bước 5 — Tổng Hợp Bảng Kết Quả và Biểu Đồ Báo Cáo

In [ ]:
# Tạo file results.xlsx đầy đủ 7 sheets
from code.build_results import build_results_excel
build_results_excel('results.xlsx')
print('Đã tạo thành công results.xlsx!')

# Tạo các biểu đồ bổ trợ
from code.generate_report_charts import main as gen_charts
gen_charts()
print('Đã tạo thành công các biểu đồ trong curves/!')
